# **Parte 1 – Recolección de noticias** 

## Paso 1: Configuración inicial

**Pregunta guía:** ¿Pueden convertir titulares peruanos en un clasificador de noticias?

| Dato | Valor |
|---|---|
| Fecha de descarga | 2026-10-18 |
| Opción | B – El Comercio |
| robots.txt | https://elcomercio.pe/robots.txt |
| Sitemaps | https://elcomercio.pe/sitemap/news/{seccion}/?outputType=xml |
| Secciones | politica, economia, tecnologia, deporte-total |
| Bajada | etiqueta `<meta name="description">` de cada noticia |

**Por qué El Comercio.** Revisé La República: su RSS trae solo 12–34 noticias por sección
y su robots.txt prohíbe la paginación (`Disallow: /*/pag/`). El Comercio publica en su
robots.txt un sitemap de noticias por sección con las 100 más recientes.
**Buenas prácticas:** User-Agent que identifica el proyecto, pausa de 1.5 s entre
solicitudes, máximo 1 reintento y ninguna ruta prohibida.


In [23]:
import time
from pathlib import Path
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

import requests
from bs4 import BeautifulSoup
import pandas as pd
import plotly.express as px
import plotly.io as pio

# Interactivo en VS Code y, además, como imagen PNG para que se vea en GitHub
pio.renderers.default = "plotly_mimetype+png"

In [24]:
FECHA_DESCARGA = "2026-10-17"
MEDIO = "El Comercio"
SECCIONES = ["politica", "economia", "tecnologia", "deporte-total"]

ENCABEZADOS = {
    "User-Agent": "TareaPUCP-clasificador-noticias/1.0 (proyecto academico; github.com/NoeliaLRuiz)"
}
PAUSA_SEGUNDOS = 1.5

CARPETA_DATOS = Path("datos")
CARPETA_CRUDOS = CARPETA_DATOS / "raw"
CARPETA_CRUDOS.mkdir(parents=True, exist_ok=True)

### Revisión de robots.txt
Antes de descargar, reviso qué permite El Comercio. Su robots.txt permite leer los sitemaps de noticias y prohíbe, por ejemplo, el buscador (`/buscar/`). La celda lo comprueba: debe responder `True` para el sitemap y `False` para el buscador.

In [25]:
respuesta = requests.get("https://elcomercio.pe/robots.txt", headers=ENCABEZADOS, timeout=20)
robots = RobotFileParser()
robots.parse(respuesta.text.splitlines())

agente = ENCABEZADOS["User-Agent"]
url_sitemap = "https://elcomercio.pe/sitemap/news/politica/?outputType=xml"
url_buscador = "https://elcomercio.pe/buscar/congreso/"
print("¿Puedo leer el sitemap?", robots.can_fetch(agente, url_sitemap))
print("¿Puedo usar el buscador?", robots.can_fetch(agente, url_buscador))

¿Puedo leer el sitemap? True
¿Puedo usar el buscador? False


## Paso 2: Extracción

### Descarga de noticias
Todas las solicitudes pasan por una sola función, `descargar()`, para que cumplan las mismas reglas: se identifican con un User-Agent propio, esperan 1.5 segundos después de cada
solicitud y reintentan una sola vez si algo falla.


In [26]:
def descargar(url):
    """Descarga una página con pausa y como máximo 1 reintento."""
    for intento in [1, 2]:
        try:
            respuesta = requests.get(url, headers=ENCABEZADOS, timeout=20)
            time.sleep(PAUSA_SEGUNDOS)
            if respuesta.status_code == 200:
                return respuesta.text
            print("Código", respuesta.status_code, "en", url)
        except requests.RequestException as error:
            print("Error en el intento", intento, ":", error)
            time.sleep(PAUSA_SEGUNDOS)
    return None

### Sitemaps de noticias
El Comercio publica un sitemap de noticias por sección con las 100 más recientes (titular, URL y fecha). Descargo los de política, economía, tecnología y deporte, y guardo cada XML en `datos/raw/` como respaldo crudo. Si el archivo ya existe, no se vuelve a descargar: así, al ejecutar de nuevo el notebook, los datos no cambian.

In [27]:
for seccion in SECCIONES:
    archivo = CARPETA_CRUDOS / f"sitemap_{seccion}_{FECHA_DESCARGA}.xml"
    if archivo.exists():
        print("Ya estaba descargado:", archivo.name)
        continue
    texto = descargar(f"https://elcomercio.pe/sitemap/news/{seccion}/?outputType=xml")
    if texto is not None:
        archivo.write_text(texto, encoding="utf-8")
        print("Guardado:", archivo.name)

Ya estaba descargado: sitemap_politica_2026-10-17.xml
Ya estaba descargado: sitemap_economia_2026-10-17.xml
Ya estaba descargado: sitemap_tecnologia_2026-10-17.xml
Ya estaba descargado: sitemap_deporte-total_2026-10-17.xml


In [28]:
filas = []
for archivo in sorted(CARPETA_CRUDOS.glob("sitemap_*.xml")):
    sopa = BeautifulSoup(archivo.read_text(encoding="utf-8"), "xml")
    for noticia in sopa.find_all("url"):
        titular = noticia.find("news:title")
        fecha = noticia.find("news:publication_date")
        filas.append({
            "url": noticia.find("loc").get_text(strip=True),
            "titular": titular.get_text(strip=True) if titular else "",
            "fecha": fecha.get_text(strip=True)[:10] if fecha else "",
        })

noticias_crudas = pd.DataFrame(filas)
print(noticias_crudas.shape)
noticias_crudas.head()

(400, 3)


,url,titular,fecha
0,https://elcomercio.pe/deporte-total/futbol-per...,El furioso “Que se dedique a administrar” de G...,2026-10-18
1,https://elcomercio.pe/deporte-total/futbol-per...,“El jugador peruano tiene un déficit grande en...,2026-10-18
2,https://elcomercio.pe/deporte-total/futbol-per...,“¿A quién le sirve un VAR que funciona con tan...,2026-10-18
3,https://elcomercio.pe/deporte-total/futbol-per...,Alianza Lima vs. Atlético Grau en vivo: minuto...,2026-10-17
4,https://elcomercio.pe/deporte-total/futbol-per...,¡Golazo! Vélez desparramó rivales y anotó el 3...,2026-10-17


Resultado: 400 noticias (100 por sitemap), publicadas entre el 6 y el 18 de octubre de 2026. Una misma URL aparece en dos sitemaps; se elimina en la limpieza.

### Bajadas
El sitemap no trae la bajada. La tomo de la etiqueta `<meta name="description">` de cada noticia, visitándolas una por una (399 páginas, unos 15 minutos). El avance se guarda cada
25 noticias en `datos/raw/bajadas.csv`, para retomar si la descarga se corta. No guardo el HTML completo de cada página porque pesaría unos 200 MB.

In [29]:
archivo_bajadas = CARPETA_CRUDOS / "bajadas.csv"
if archivo_bajadas.exists():
    bajadas = pd.read_csv(archivo_bajadas)
else:
    bajadas = pd.DataFrame(columns=["url", "bajada"])

urls_ya_visitadas = set(bajadas["url"])
urls_pendientes = []
for url in noticias_crudas["url"].drop_duplicates():
    if url not in urls_ya_visitadas:
        urls_pendientes.append(url)
print("Noticias por visitar:", len(urls_pendientes))

Noticias por visitar: 0


In [30]:
nuevas = []
for numero, url in enumerate(urls_pendientes, start=1):
    html = descargar(url)
    bajada = ""
    if html is not None:
        sopa = BeautifulSoup(html, "lxml")
        meta = sopa.find("meta", attrs={"name": "description"})
        if meta is not None:
            bajada = meta.get("content", "").strip()
    nuevas.append({"url": url, "bajada": bajada})

    if numero % 25 == 0 or numero == len(urls_pendientes):
        bajadas = pd.concat([bajadas, pd.DataFrame(nuevas)], ignore_index=True)
        bajadas.to_csv(archivo_bajadas, index=False, encoding="utf-8")
        nuevas = []
        print("Avance:", numero, "de", len(urls_pendientes))

In [31]:
print("Bajadas guardadas en total:", len(bajadas))

Bajadas guardadas en total: 399


Resultado: 399 bajadas, una por cada URL distinta, ninguna vacía.

### Tabla de trabajo
Uno titulares y bajadas por URL, agrego el nombre del medio y saco la sección original de la ruta de la URL (por ejemplo, `politica/congreso`). Con esa sección se hace el mapeo a las 4 categorías en la limpieza.

In [32]:
bajadas = bajadas.drop_duplicates(subset="url")
noticias = noticias_crudas.merge(bajadas, on="url", how="left")
noticias["bajada"] = noticias["bajada"].fillna("")
noticias["medio"] = MEDIO

def seccion_desde_url(url):
    """'https://elcomercio.pe/politica/congreso/titulo-noticia/' -> 'politica/congreso'"""
    partes = urlparse(url).path.strip("/").split("/")
    return "/".join(partes[:-1])

noticias["seccion_original"] = noticias["url"].apply(seccion_desde_url)
noticias["seccion_original"].value_counts()

seccion_original
economia/peru                         46
tecnologia/actualidad                 37
deporte-total/futbol-peruano          31
politica/congreso                     27
politica                              24
politica/gobierno                     20
tecnologia/inteligencia-artificial    19
deporte-total/futbol-mundial          19
economia/mercados                     18
deporte-total                         16
economia                              14
tecnologia/ciencias                   11
economia/dia-1                        10
economia/opinion                       9
politica/actualidad                    8
deporte-total/argentina                8
deporte-total/polideportivo            8
tecnologia/ciberseguridad              7
tecnologia/ecologia                    6
tecnologia/videojuegos                 5
politica/internacional                 5
tecnologia/moviles                     4
deporte-total/espana                   4
ecdata                                 4